In [451]:
import torch
from torch.distributions.multivariate_normal import MultivariateNormal
import math
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def lorenz96_rhs(x, F = 8.0):
    """
    Lorenz-96 vector field.

    x: Tensor of shape (..., N)
    F: scalar or tensor broadcastable to x[..., 0]

    Returns: dx/dt with same shape as x
    """
    # Periodic neighbors using roll
    x_ip1 = torch.roll(x, shifts=-1, dims=-1)  # x_{i+1}
    x_im1 = torch.roll(x, shifts=+1, dims=-1)  # x_{i-1}
    x_im2 = torch.roll(x, shifts=+2, dims=-1)  # x_{i-2}

    # dx_i/dt = (x_{i+1} - x_{i-2}) * x_{i-1} - x_i + F
    return (x_ip1 - x_im2) * x_im1 - x + torch.as_tensor(F, device=x.device, dtype=x.dtype)


def rk4_step(
    x,
    dt,
    rhs=lorenz96_rhs,
    rhs_args = (),
    rhs_kwargs = None,
):
    """
    One Runge-Kutta 4 step: x_{n+1} = x_n + dt/6*(k1 + 2k2 + 2k3 + k4).
    """
    if rhs_kwargs is None:
        rhs_kwargs = {}

    k1 = rhs(x, *rhs_args, **rhs_kwargs)
    k2 = rhs(x + 0.5 * dt * k1, *rhs_args, **rhs_kwargs)
    k3 = rhs(x + 0.5 * dt * k2, *rhs_args, **rhs_kwargs)
    k4 = rhs(x + dt * k3, *rhs_args, **rhs_kwargs)

    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


@torch.no_grad()
def integrate_lorenz96(
    x0,
    dt,
    steps,
    F = 8.0,
    save_every= 1,
):
    """
    Integrate Lorenz-96 forward with RK4.

    x0: (..., N) initial condition
    dt: timestep
    steps: number of RK4 steps
    F: forcing
    save_every: store every k-th step (including step 0)

    Returns:
      traj: (T, ..., N) where T = steps//save_every + 1
    """
    x = x0.clone()
    out = [x0.clone()]
    

    rhs_kwargs = {"F": F}
    for s in range(1, steps + 1):
        x = rk4_step(x, dt, rhs=lorenz96_rhs, rhs_kwargs=rhs_kwargs)
        if (s % save_every) == 0:
            out.append(x.clone())

    return torch.stack(out, dim=0)


# Example usage:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float32

    B, N = 100, 40
    x0 = torch.randn(B, N, device=device, dtype=dtype)
    T = 1000
    traj = integrate_lorenz96(x0, dt=0.01, steps=T, F=8.0, save_every=1)
    print(traj.shape)  # (101, 16, 40)

torch.Size([1001, 100, 40])


In [576]:
#Define trajectory shapes T= number of time steps, B= number of training trajectories, N=dimension of Lorenz 96
T, B, N = traj.shape
#Forecast horizon
horizon = 1
#Window L s.t. x_{t+1} = [x_t,...,x_{t-L+1}]
L = 40
#Weight decay rate
gamma = 0.9
p = 8

#t_0 minimum 'valid' time for time window
t0 = L - 1
T_train = int(T/2)
#maximum valid time
t1 = T_train- 1 - horizon
t_vals = list(range(t0, t1+1))
#valid time length
n_t = len(t_vals)

traj_train = traj[:T_train, :, :]                    # (T,B,p)

# standardize observations (inputs only), using TRAIN trajectories ideally
mu_u = traj_train.mean(dim=(0,1), keepdim=True)   # (1,1,p)
sd_u = traj_train.std(dim=(0,1), keepdim=True) + 1e-6
std = (traj_train - mu_u) / sd_u                # (T,B,p)


# lag weights, most recent first: [w0 for o_t, w1 for o_{t-1}, ...]
w = (gamma ** torch.arange(L, device=traj.device, dtype=traj.dtype))  # (L,)

#Initialize targets Y and windows U
Y = torch.zeros(B, n_t, N,   device=traj.device, dtype=traj.dtype)
U = torch.zeros(B, n_t, N*L, device=traj.device, dtype=traj.dtype)
U_weight = torch.zeros(B, n_t, N*L, device=traj.device, dtype=traj.dtype)
for j in range(B):
    for i, t in enumerate(t_vals):
        window = std[t-L+1:t+1, j, :]        # (L,p)
        window = torch.flip(window, dims=[0])    # newest->oldest
        U[j, i, :] = window.reshape(-1)
        window = window * w[:, None]
        U_weight[j, i, :] = window.reshape(-1)
        Y[j, i, :] = traj_train[t + horizon, j, :] - traj_train[t, j, :]

# # flatten across trajectories/time
U_flat = U.reshape(B*n_t, N*L)
U_weight_flat = U_weight.reshape(B*n_t, N*L)
Y_flat = Y.reshape(B*n_t, N)
#        window = window * w[:, None]
sample_size = U_flat.shape[0]
idx = torch.randperm(U_flat.shape[0], device=U_flat.device)[:sample_size]
#Sample trajectories
U_train = U_flat[idx]
U_weight_train = U_weight_flat[idx]
Y_train_raw = Y_flat[idx]
#Standardize targets
Y_mean = Y_train_raw.mean(dim=0, keepdim=True)
Y_std  = Y_train_raw.std(dim=0, keepdim=True) + 1e-6

Y_train = (Y_train_raw - Y_mean) / Y_std

In [526]:
U_flat.shape

torch.Size([46000, 1600])

In [527]:
def phi(u, Omega, b, sigma=1.0, P=512):
    # u: (d_u,)
    z = math.sqrt(2.0 / P) * torch.cos(Omega @ u + b)  # (P,)
    return torch.cat([u, sigma * z], dim=0)            # (d_u+P,)

def median_ell(X, sample_size=5000):
    n = X.shape[0]
    idx1 = torch.randint(0, n, (sample_size,), device=X.device)
    idx2 = torch.randint(0, n, (sample_size,), device=X.device)
    same = idx1 == idx2
    while same.any():
        idx2[same] = torch.randint(0, n, (same.sum(),), device=X.device)
        same = idx1 == idx2
    d = torch.linalg.norm(X[idx1] - X[idx2], dim=1)
    return d.median()


def solve_W(X, Y, Omega, b,sigma = 1.0,P=512, lam=10**3):
        # U: (n, d_u), Y: (n, p_out)
    n, d_u = X.shape
    p_out = Y.shape[1]
    d = d_u + P

    G = torch.zeros(d, d, device=X.device, dtype=X.dtype)
    H = torch.zeros(d, p_out, device=X.device, dtype=X.dtype)

    for i in range(n):
        phi_i = phi(X[i], Omega, b, sigma, P)# (d,)
        G += torch.outer(phi_i, phi_i)                     # (d,d)
        H += torch.outer(phi_i, Y[i])                      # (d,p_out)
    G /= n
    H /= n

    W = torch.linalg.solve(G + lam * torch.eye(d, device=X.device, dtype=X.dtype), H)  # (d,p_out)
    
    return W


def model_mse(y_true, y_pred):
    return ((y_true - y_pred)**2).mean()


In [528]:
def phi_batch(U, Omega, b, sigma=1.0, P=512):
    """
    U: (B, d_u)
    Omega: (P, d_u)
    b: (P,)
    returns: (B, d_u + P)
    """
    Z = math.sqrt(2.0 / P) * torch.cos(U @ Omega.T + b)   # (B, P)
    return torch.cat([U, sigma * Z], dim=1)               # (B, d_u + P)



def rollout_beyond_T_batch(
    traj, T_train, Omega, b, sigma, p, P, W,
    U_mean, U_std, Y_mean, Y_std, L, steps,w
):
    """
    traj: (T_total, B, p)
    W should be trained for one-step increments.
    returns:
        future: (steps+1, B, p) containing
                [x_T, x_{T+1}^hat, ..., x_{T+steps}^hat]
    """
    device = traj.device
    dtype = traj.dtype
    B = traj.shape[1]

    # Initialize with true window ending at T_train
    
    hist = traj[T_train - L:T_train, :, :p].clone()   

    future = torch.empty(steps + 1, B, p, device=device, dtype=dtype)
    future[0] = hist[-1]   # true state at T_train

    for k in range(steps):
        hist_std = (hist - U_mean) / U_std                     # (L, B, p)
        window = torch.flip(hist_std, dims=[0])*w[:, None, None]             # newest->oldest
        U = window.permute(1, 0, 2).reshape(B, L * p)         # (B, Lp)

        do_std = phi_batch(U, Omega, b, sigma, P) @ W         # (B, p)
        do = Y_mean + Y_std * do_std                          # (B, p)

        o_t = hist[-1]                                        # (B, p)
        o_next = o_t + do                                     # (B, p)

        future[k + 1] = o_next

        # roll window forward
        hist = torch.cat([hist[1:], o_next.unsqueeze(0)], dim=0)

    return future

In [529]:
#####No forecast PCA######

ell_u = median_ell(U_weight_train)
sigma_u = 1
#Number of random Fourier features
P_u = 512
#Generate numbers from normal distribution 
Omega_u = torch.randn(P_u, U_weight_train.shape[1], device=U_weight_train.device, dtype=traj.dtype) / ell_u
#Generate randoms from uniform distribution
b_u = (2*torch.pi*torch.rand((P_u,), device=U_weight_train.device, dtype=dtype))

W_u = solve_W(U_weight_train, Y_train, Omega_u, b_u, sigma=1.0, P=P_u, lam=10**-2)

In [530]:
median_ell(U_train)

tensor(57.5652)

In [531]:
steps = 10
future_pred = rollout_beyond_T_batch(traj, T_train, Omega_u, b_u, sigma_u, N, P_u, W_u,
                                     mu_u, sd_u, Y_mean, Y_std, L,steps=steps,w=w)

In [532]:
torch.linalg.norm(future_pred[1:,:,:]-traj[T_train+1:T_train+steps+1,:,:])/torch.linalg.norm(traj[T_train+1:T_train+steps+1,:,:])

tensor(0.1156)

In [533]:
errs = []
coss = []
inc_coss = []

for k in range(1, steps + 1):
    x_true = traj[T_train - 1 + k]      # adjust if you shift indexing
    x_pred = future_pred[k]

    rel = torch.linalg.norm(x_pred - x_true) / torch.linalg.norm(x_true)
    errs.append(rel.item())

    cos = torch.sum(x_pred * x_true) / (torch.linalg.norm(x_pred) * torch.linalg.norm(x_true) + 1e-12)
    coss.append(cos.item())

for k in range(1, steps):
    dx_true = traj[T_train - 1 + k + 1] - traj[T_train - 1 + k]
    dx_pred = future_pred[k + 1] - future_pred[k]

    inc_cos = torch.sum(dx_pred * dx_true) / (torch.linalg.norm(dx_pred) * torch.linalg.norm(dx_true) + 1e-12)
    inc_coss.append(inc_cos.item())


In [534]:
print(errs)
print(inc_coss)

[0.010148106142878532, 0.02102048136293888, 0.03303961828351021, 0.04646671935915947, 0.061440013349056244, 0.07800274342298508, 0.09612400084733963, 0.11571431905031204, 0.13663823902606964, 0.15872463583946228]
[0.9673788547515869, 0.959162175655365, 0.9477705955505371, 0.9334924817085266, 0.9166964292526245, 0.8978698253631592, 0.8776279091835022, 0.8566839098930359, 0.8358061909675598]


In [535]:
######Linear PCA#######

#Dimension of PCA
# k = 40

#Concatenate time windows across trajectories
# X_train = traj_train.reshape(B*T_train, N).detach().cpu().numpy()

# mu_x_train = traj_train.mean(dim=(0,1), keepdim=True)
# sd_x_train = traj_train.std(dim=(0,1), keepdim=True) + 1e-6
# traj_train_std = (traj_train - mu_x_train) / sd_x_train
# X_train_std = traj_train_std.reshape(T_train * B, N).detach().cpu().numpy()
# pca = PCA(n_components=k)
# S_all = pca.fit_transform(X_train_std)
# S_seq = torch.tensor(S_all, device=traj.device, dtype=traj.dtype).reshape(T_train, B, k)
# mu_pca = torch.tensor(pca.mean_, device=traj.device, dtype=traj.dtype)
# V_k = torch.tensor(pca.components_, device=traj.device, dtype=traj.dtype)
# 3) build pairs
# S_flat      = S_seq[:, :-1, :].reshape(B*(T_train-1), k)

In [536]:
#Differences
# dS_h = (S_seq[:, horizon:, :] - S_seq[:, :-horizon, :]).reshape(-1, k)

# S_train = S_seq[:, :-horizon, :].reshape(-1, k)        

# # O_seq = std[t_vals, :, :]          
# # O_seq = O_seq.permute(1, 0, 2)        
# # O = O_seq[:, :-1, :].reshape(-1, N)    

# S_mean = S_train.mean(dim=0, keepdim=True)
# S_std  = S_train.std(dim=0, keepdim=True) + 1e-6

# S_seq_std = (S_seq - S_mean) / S_std
# S_train_std = S_seq_std[:, :-horizon, :].reshape(-1, k)
# dS_h_std = (S_seq_std[:, horizon:, :] - S_seq_std[:, :-horizon, :]).reshape(-1, k)
# #S_seq_std_T = S_seq_std.permute(1, 0, 2)   # (n_t, B, k)
# S0_std = S_seq_std[T_train-1]                      # (B, k)
# dS_mean = dS_h_std.mean(dim=0, keepdim=True)
# dS_std  = dS_h_std.std(dim=0, keepdim=True) + 1e-6

In [537]:
# ell_s = median_ell(S_train_std)
# sigma_s = 1
# #Number of random Fourier features
# P_s = 512
# #Generate numbers from normal distribution 
# Omega_s = torch.randn(P_s, S_train_std.shape[1], device=S_train.device, dtype=dtype) / ell_s
# #Generate randoms from uniform distribution
# b_s = (2*torch.pi*torch.rand((P_s,), device=S_train.device, dtype=dtype))
# W_s = solve_W(S_train_std, dS_h_std, Omega_s, b_s,sigma_s,P_s,10**-5)


In [538]:
# def rollout_state_space(S0_std, Omega, b, sigma, P, W, Y_mean, Y_std, steps):
#     """
#     S0_std: (B, k) initial latent state in standardized PCA coordinates
#     returns:
#         future: (steps+1, B, k)
#     """
#     device = S0_std.device
#     dtype = S0_std.dtype
#     B, k = S0_std.shape

#     future = torch.empty(steps + 1, B, k, device=device, dtype=dtype)
#     future[0] = S0_std

#     s = S0_std
#     for t in range(steps):
#         ds_std = phi_batch(s, Omega, b, sigma, P) @ W   # (B, k)
#         ds = Y_mean + Y_std * ds_std                    # if targets were standardized
#         s = s + ds
#         future[t + 1] = s

#     return future

In [539]:
# S_hat_std = rollout_state_space(
#     S0_std, Omega_s, b_s, sigma_s, P_s, W_s, dS_mean, dS_std, steps
# )

In [540]:
# def decode_pca_rollout(S_hat_std, S_mean, S_std, mu_pca, V_k, mu_x, sd_x, B, L, N):
#     """
#     S_hat_std: (steps+1, B, k) standardized latent rollout
#     S_mean, S_std: (1, k) or broadcastable latent normalization stats
#     mu_pca: (N*L,) PCA mean in flattened standardized-window space
#     V_k: (k, N*L) PCA components
#     mu_x, sd_x: raw state standardization stats, broadcastable to (steps+1,B,N)
#     returns:
#         X_hat:      (steps+1, B, N) decoded states in original scale
#         X_hat_std:  (steps+1, B, N) decoded states in standardized scale
#         U_hat_std:  (steps+1, B, L, N) decoded windows in standardized scale
#     """
#     S_hat = S_mean + S_std * S_hat_std                          # (steps+1, B, k)
#     U_hat_flat_std = mu_pca + torch.einsum('tbk,kn->tbn', S_hat, V_k)  # (steps+1, B, N*L)
#     U_hat_std = U_hat_flat_std.reshape(S_hat.shape[0], B, L, N)        # (steps+1, B, L, N)

#     # newest state is at index 0 because you flipped newest->oldest before flattening
#     X_hat_std = U_hat_std[:, :, 0, :]                           # (steps+1, B, N)
#     X_hat = mu_x + sd_x * X_hat_std

#     return X_hat, X_hat_std, U_hat_std

In [541]:
# X_hat,X_hat_std,_ = decode_pca_rollout(S_hat_std=S_hat_std,S_mean=S_mean,S_std=S_std,mu_pca=mu_pca,V_k=V_k,mu_x=mu_x_train,sd_x=sd_x_train,B=B,L=L,N=N)

In [542]:
def rollout_state_space_snapshot_std(S0_std, Omega, b, sigma, P, W, steps):
    future = torch.empty(steps + 1, *S0_std.shape, device=S0_std.device, dtype=S0_std.dtype)
    future[0] = S0_std
    s = S0_std
    for t in range(steps):
        ds = phi_batch(s, Omega, b, sigma, P) @ W
        s = s + ds
        future[t + 1] = s
    return future



In [543]:
# training split
traj_train = traj[:T_train]                       # (T_train, B, N)

# standardize raw states on training set
mu_x_train = traj_train.mean(dim=(0,1), keepdim=True)
sd_x_train = traj_train.std(dim=(0,1), keepdim=True) + 1e-6
traj_train_std = (traj_train - mu_x_train) / sd_x_train

# PCA on standardized snapshots
X_train_std = traj_train_std.reshape(T_train * B, N).detach().cpu().numpy()
k = 40
pca = PCA(n_components=k)
S_all = pca.fit_transform(X_train_std)

S_seq = torch.tensor(S_all, device=traj.device, dtype=traj.dtype).reshape(T_train, B, k)
mu_pca = torch.tensor(pca.mean_, device=traj.device, dtype=traj.dtype)      # in standardized x-space
V_k = torch.tensor(pca.components_, device=traj.device, dtype=traj.dtype)   # (k, N)

# latent standardization
S_train = S_seq[:-1].reshape(-1, k)
S_mean = S_train.mean(dim=0, keepdim=True)
S_std = S_train.std(dim=0, keepdim=True) + 1e-6

S_seq_std = (S_seq - S_mean) / S_std
S_train_std = S_seq_std[:-1].reshape(-1, k)
dS_train_std = (S_seq_std[1:] - S_seq_std[:-1]).reshape(-1, k)

# fit latent increment model
P_s = 512
sigma_s = 1.0
ell_s = median_ell(S_train_std)
Omega_s = torch.randn(P_s, k, device=traj.device, dtype=traj.dtype) / ell_s
b_s = 2 * torch.pi * torch.rand(P_s, device=traj.device, dtype=traj.dtype)
W_s = solve_W(S_train_std, dS_train_std, Omega_s, b_s, sigma_s, P_s, 1e-2)

# rollout from last training state
S0_std = S_seq_std[T_train - 1]   # (B, k)
future_pred_snapshot = rollout_state_space_snapshot_std(S0_std, Omega_s, b_s, sigma_s, P_s, W_s, steps)

# decode
S_hat = S_mean + S_std * future_pred_snapshot
X_hat_std = mu_pca + torch.einsum('tbk,kn->tbn', S_hat, V_k)
X_hat = mu_x_train + sd_x_train * X_hat_std

# truth for forecast times T_train, ..., T_train+steps-1
traj_future = traj[T_train:T_train+steps]
traj_future_std = (traj_future - mu_x_train) / sd_x_train

errs_pca = []
for h in range(1, steps + 1):
    errs_pca.append(
        (torch.linalg.norm(X_hat_std[h] - traj_future_std[h-1]) /
         torch.linalg.norm(traj_future_std[h-1])).item()
    )

In [544]:
def build_delay_windows(traj_std, L, gamma=1.0):
    """
    traj_std: (T, B, N) standardized states
    returns:
        U_unweighted: (B, n_t, L*N) flattened windows, newest->oldest
        U_weighted:   (B, n_t, L*N) weighted flattened windows
        t_vals:       list of window-end times
    """
    T, B, N = traj_std.shape
    t0 = L - 1
    t1 = T - 1
    t_vals = list(range(t0, t1 + 1))
    n_t = len(t_vals)

    w = gamma ** torch.arange(L, device=traj_std.device, dtype=traj_std.dtype)

    U_unweighted = torch.empty(B, n_t, L * N, device=traj_std.device, dtype=traj_std.dtype)
    U_weighted   = torch.empty(B, n_t, L * N, device=traj_std.device, dtype=traj_std.dtype)

    for j in range(B):
        for i, t in enumerate(t_vals):
            window = traj_std[t-L+1:t+1, j, :]          # (L, N), oldest->newest
            window = torch.flip(window, dims=[0])       # newest->oldest
            U_unweighted[j, i, :] = window.reshape(-1)
            U_weighted[j, i, :] = (window * w[:, None]).reshape(-1)

    return U_unweighted, U_weighted, t_vals


def rollout_state_space_window_pca(
    S0_std, Omega, b, sigma, P, W, steps
):
    """
    Roll out in standardized latent coordinates.
    Assumes W predicts the one-step latent increment in standardized latent coords.

    S0_std: (B, k)
    returns:
        S_future_std: (steps+1, B, k)
    """
    device = S0_std.device
    dtype = S0_std.dtype
    B, k = S0_std.shape

    S_future_std = torch.empty(steps + 1, B, k, device=device, dtype=dtype)
    S_future_std[0] = S0_std

    s = S0_std
    for h in range(steps):
        ds = phi_batch(s, Omega, b, sigma, P) @ W   # (B, k)
        s = s + ds
        S_future_std[h + 1] = s

    return S_future_std


def decode_window_pca_rollout(
    S_future_std, S_mean, S_std, mu_pca, V_k, mu_x, sd_x, L, N
):
    """
    Decode latent rollout back to standardized windows, then extract newest state.

    Inputs:
        S_future_std: (steps+1, B, k)
        S_mean, S_std: (1, k) latent normalization stats
        mu_pca: (L*N,) PCA mean in standardized window space
        V_k: (k, L*N) PCA components from sklearn
        mu_x, sd_x: (1,1,N) raw-state standardization stats
    Returns:
        X_hat:        (steps+1, B, N) decoded states in original scale
        X_hat_std:    (steps+1, B, N) decoded states in standardized scale
        U_hat_std:    (steps+1, B, L, N) decoded windows in standardized scale
        U_hat_flat:   (steps+1, B, L*N) decoded flattened windows in standardized scale
    """
    # undo latent standardization
    S_future = S_mean + S_std * S_future_std                       # (steps+1, B, k)

    # decode PCA window coordinates
    U_hat_flat = mu_pca + torch.einsum('tbk,kn->tbn', S_future, V_k)   # (steps+1, B, L*N)
    U_hat_std = U_hat_flat.reshape(S_future.shape[0], S_future.shape[1], L, N)

    # newest state is index 0 because windows were stored newest->oldest
    X_hat_std = U_hat_std[:, :, 0, :]                               # (steps+1, B, N)
    X_hat = mu_x + sd_x * X_hat_std

    return X_hat, X_hat_std, U_hat_std, U_hat_flat


# ============================================================
# CLEAN WINDOW-PCA PIPELINE
# ============================================================

# traj: (T, B, N)
T, B, N = traj.shape
L = 40
gamma = 1.0
horizon = 1          # must be 1 for recursive rollout
steps = 10
k = 150              # choose <= L*N and <= number of train windows

T_train = T // 2

# 1) training block and standardization on training block only
traj_train = traj[:T_train]                                         # (T_train, B, N)
mu_x_train = traj_train.mean(dim=(0, 1), keepdim=True)              # (1,1,N)
sd_x_train = traj_train.std(dim=(0, 1), keepdim=True) + 1e-6        # (1,1,N)

traj_train_std = (traj_train - mu_x_train) / sd_x_train             # (T_train, B, N)

# 2) build windows on training block
U_unw, U_w, t_vals = build_delay_windows(traj_train_std, L=L, gamma=gamma)
# use weighted or unweighted consistently; with gamma=1 they are identical
U_train_windows = U_w                                                # (B, n_t, L*N)

B_, n_t, D = U_train_windows.shape
assert B_ == B and D == L * N

# 3) fit PCA on training windows only
U_flat = U_train_windows.reshape(B * n_t, D).detach().cpu().numpy()
pca = PCA(n_components=k)
S_all = pca.fit_transform(U_flat)                                    # (B*n_t, k)

# IMPORTANT: reshape back consistently with the flattening above
S_seq = torch.tensor(S_all, device=traj.device, dtype=traj.dtype).reshape(B, n_t, k)

mu_pca = torch.tensor(pca.mean_, device=traj.device, dtype=traj.dtype)        # (L*N,)
V_k = torch.tensor(pca.components_, device=traj.device, dtype=traj.dtype)     # (k, L*N)

# 4) latent one-step pairs
# S_seq[:, i, :] corresponds to window ending at t_vals[i]
S_train = S_seq[:, :-1, :].reshape(-1, k)                             # current latent state
dS = (S_seq[:, 1:, :] - S_seq[:, :-1, :]).reshape(-1, k)             # one-step latent increment

# 5) standardize latent states (and work directly with dS in std-latent coordinates)
S_mean = S_train.mean(dim=0, keepdim=True)
S_std = S_train.std(dim=0, keepdim=True) + 1e-6

S_seq_std = (S_seq - S_mean) / S_std
S_train_std = S_seq_std[:, :-1, :].reshape(-1, k)
dS_std = (S_seq_std[:, 1:, :] - S_seq_std[:, :-1, :]).reshape(-1, k)

# 6) fit one-step latent increment model
dtype = traj.dtype
ell_s = median_ell(S_train_std)
sigma_s = 1.0
P_s = 512

Omega_s = torch.randn(P_s, k, device=traj.device, dtype=dtype) / ell_s
b_s = 2 * torch.pi * torch.rand((P_s,), device=traj.device, dtype=dtype)

# solve_W should use the targets as given; here targets are already dS_std
W_s = solve_W(S_train_std, dS_std, Omega_s, b_s, sigma=sigma_s, P=P_s, lam=1e-2)

# 7) initialize from the last training window
# last training window ends at time T_train-1
# t_vals[-1] should equal T_train-1
assert t_vals[-1] == T_train - 1

S0_std = S_seq_std[:, -1, :]                                         # (B, k)

# 8) latent rollout
S_future_std = rollout_state_space_window_pca(
    S0_std, Omega_s, b_s, sigma_s, P_s, W_s, steps
)

# 9) decode back to current state
X_hat, X_hat_std, U_hat_std, U_hat_flat = decode_window_pca_rollout(
    S_future_std, S_mean, S_std, mu_pca, V_k, mu_x_train, sd_x_train, L, N
)

# 10) evaluate against held-out future
# X_hat_std[0] corresponds to time T_train-1 (decoded from last training window)
# X_hat_std[1] corresponds to predicted time T_train
traj_future = traj[T_train:T_train + steps, :, :]                     # truth at times T_train ... T_train+steps-1
traj_future_std = (traj_future - mu_x_train) / sd_x_train

errs_pca_window = []
for h in range(1, steps + 1):
    rel_err = (
        torch.linalg.norm(X_hat_std[h] - traj_future_std[h - 1]) /
        torch.linalg.norm(traj_future_std[h - 1])
    ).item()
    errs_pca_window.append(rel_err)

print(errs_pca_window)

[0.08070371299982071, 0.10401199758052826, 0.12981782853603363, 0.1579582393169403, 0.18825624883174896, 0.2205154299736023, 0.25451961159706116, 0.29003530740737915, 0.32681503891944885, 0.36460021138191223]


In [552]:
def fourier_encode_snapshot(x_std, K):
    """
    x_std: (..., N) real standardized state on ring
    K: number of positive frequencies to keep (exclude Nyquist unless you want it explicitly)

    returns:
        s: (..., 1 + 2K) real latent Fourier coordinates
           [Re(fft[0]), Re(fft[1]), Im(fft[1]), ..., Re(fft[K]), Im(fft[K])]
    """
    Xf = torch.fft.rfft(x_std, dim=-1)   # (..., N//2 + 1), complex
    parts = [Xf[..., 0].real.unsqueeze(-1)]   # DC mode

    for k in range(1, K + 1):
        parts.append(Xf[..., k].real.unsqueeze(-1))
        parts.append(Xf[..., k].imag.unsqueeze(-1))

    s = torch.cat(parts, dim=-1)
    return s


def fourier_decode_snapshot(s, N, K):
    """
    s: (..., 1 + 2K) real latent Fourier coordinates
    N: original state dimension
    K: number of positive frequencies retained

    returns:
        x_std_hat: (..., N) reconstructed standardized state
    """
    shape_prefix = s.shape[:-1]
    device = s.device
    dtype = s.dtype

    # rfft shape for real signal of length N is N//2 + 1
    Xf = torch.zeros(*shape_prefix, N // 2 + 1, device=device, dtype=torch.complex64)

    # If input dtype is float64, use complex128
    if dtype == torch.float64:
        Xf = Xf.to(torch.complex128)

    Xf[..., 0] = s[..., 0].to(Xf.real.dtype)

    idx = 1
    for k in range(1, K + 1):
        re = s[..., idx]
        im = s[..., idx + 1]
        Xf[..., k] = torch.complex(re, im)
        idx += 2

    # all higher modes remain zero
    x_std_hat = torch.fft.irfft(Xf, n=N, dim=-1)
    return x_std_hat


def rollout_state_space_fourier_snapshot(
    S0_std, Omega, b, sigma, P, W, steps
):
    """
    Roll out in standardized latent Fourier coordinates.

    S0_std: (B, d)
    returns:
        S_future_std: (steps+1, B, d)
    """
    device = S0_std.device
    dtype = S0_std.dtype
    B, d = S0_std.shape

    S_future_std = torch.empty(steps + 1, B, d, device=device, dtype=dtype)
    S_future_std[0] = S0_std

    s = S0_std
    for h in range(steps):
        ds = phi_batch(s, Omega, b, sigma, P) @ W   # predicts one-step increment in standardized latent coords
        s = s + ds
        S_future_std[h + 1] = s

    return S_future_std


def decode_fourier_rollout_snapshot_std(S_future_std, S_mean, S_std, N, K, mu_x, sd_x):
    """
    S_future_std: (steps+1, B, d) rollout in standardized latent coordinates
    S_mean, S_std: (1, d) latent normalization stats
    mu_x, sd_x: (1,1,N) raw-state standardization stats

    returns:
        X_hat:     (steps+1, B, N) in original scale
        X_hat_std: (steps+1, B, N) in standardized state scale
    """
    S_future = S_mean + S_std * S_future_std
    X_hat_std = fourier_decode_snapshot(S_future, N=N, K=K)
    X_hat = mu_x + sd_x * X_hat_std
    return X_hat, X_hat_std


# ============================================================
# CLEAN FOURIER SNAPSHOT PIPELINE
# ============================================================

# traj: (T, B, N)
T, B, N = traj.shape
T_train = T // 2
steps = 10
horizon = 1   # must be 1 for recursive rollout

# choose truncation:
# latent dimension d = 1 + 2K
K = 19
d_fourier = 1 + 2 * K

traj_train = traj[:T_train]   # (T_train, B, N)

# 1) standardize raw states on training block only
mu_x_train = traj_train.mean(dim=(0, 1), keepdim=True)              # (1,1,N)
sd_x_train = traj_train.std(dim=(0, 1), keepdim=True) + 1e-6        # (1,1,N)
traj_train_std = (traj_train - mu_x_train) / sd_x_train             # (T_train, B, N)

# 2) encode every snapshot into truncated real Fourier coordinates
S_seq = fourier_encode_snapshot(traj_train_std, K=K)                # (T_train, B, d_fourier)

# 3) build one-step latent pairs
S_train = S_seq[:-1].reshape(-1, d_fourier)
dS = (S_seq[1:] - S_seq[:-1]).reshape(-1, d_fourier)

# 4) standardize latent states
S_mean = S_train.mean(dim=0, keepdim=True)
S_std = S_train.std(dim=0, keepdim=True) + 1e-6

S_seq_std = (S_seq - S_mean) / S_std
S_train_std = S_seq_std[:-1].reshape(-1, d_fourier)
dS_std = (S_seq_std[1:] - S_seq_std[:-1]).reshape(-1, d_fourier)

# 5) fit one-step latent increment model
dtype = traj.dtype
ell_s = median_ell(S_train_std)
sigma_s = 1.0
P_s = 512

Omega_s = torch.randn(P_s, d_fourier, device=traj.device, dtype=dtype) / ell_s
b_s = 2 * torch.pi * torch.rand((P_s,), device=traj.device, dtype=dtype)

W_s = solve_W(S_train_std, dS_std, Omega_s, b_s, sigma=sigma_s, P=P_s, lam=1e-2)

# 6) initialize from last training state (time T_train-1)
S0_std = S_seq_std[T_train - 1]    # (B, d_fourier)

# 7) latent rollout
S_future_std = rollout_state_space_fourier_snapshot(
    S0_std, Omega_s, b_s, sigma_s, P_s, W_s, steps
)

# 8) decode back to state space
X_hat, X_hat_std = decode_fourier_rollout_snapshot_std(
    S_future_std, S_mean, S_std, N=N, K=K, mu_x=mu_x_train, sd_x=sd_x_train
)

# 9) evaluate against held-out truth
# X_hat_std[0] corresponds to time T_train-1
# X_hat_std[1] corresponds to predicted time T_train
traj_future = traj[T_train:T_train + steps, :, :]                      # truth at times T_train ... T_train+steps-1
traj_future_std = (traj_future - mu_x_train) / sd_x_train

errs_fourier = []
for h in range(1, steps + 1):
    rel_err = (
        torch.linalg.norm(X_hat_std[h] - traj_future_std[h - 1]) /
        torch.linalg.norm(traj_future_std[h - 1])
    ).item()
    errs_fourier.append(rel_err)

print(errs_fourier)

[0.1189948096871376, 0.14015498757362366, 0.16936753690242767, 0.2029610276222229, 0.23882915079593658, 0.2757968008518219, 0.3131733238697052, 0.3505229949951172, 0.38755127787590027, 0.42404767870903015]


In [567]:

# ------------------------------------------------------------
# helpers assumed available:
#   - phi_batch(U, Omega, b, sigma=1.0, P=512)
#   - solve_W(U, Y, Omega, b, sigma=1.0, P=512, lam=1.0)
#   - median_ell(X)
# ------------------------------------------------------------

def fourier_encode_snapshot(x_std, K):
    """
    x_std: (..., N) real standardized state on ring
    K: number of positive frequencies retained, with K < N//2 for simplicity

    returns:
        s: (..., dF) where dF = 1 + 2K
           [Re(Xf[0]), Re(Xf[1]), Im(Xf[1]), ..., Re(Xf[K]), Im(Xf[K])]
    """
    Xf = torch.fft.rfft(x_std, dim=-1)  # (..., N//2 + 1), complex
    parts = [Xf[..., 0].real.unsqueeze(-1)]

    for k in range(1, K + 1):
        parts.append(Xf[..., k].real.unsqueeze(-1))
        parts.append(Xf[..., k].imag.unsqueeze(-1))

    return torch.cat(parts, dim=-1)


def fourier_decode_snapshot(s, N, K):
    """
    s: (..., dF) with dF = 1 + 2K
    returns:
        x_std_hat: (..., N)
    """
    shape_prefix = s.shape[:-1]
    device = s.device
    dtype = s.dtype

    ctype = torch.complex128 if dtype == torch.float64 else torch.complex64
    Xf = torch.zeros(*shape_prefix, N // 2 + 1, device=device, dtype=ctype)

    Xf[..., 0] = s[..., 0].to(Xf.real.dtype)

    idx = 1
    for k in range(1, K + 1):
        re = s[..., idx]
        im = s[..., idx + 1]
        Xf[..., k] = torch.complex(re, im)
        idx += 2

    x_std_hat = torch.fft.irfft(Xf, n=N, dim=-1)
    return x_std_hat


def build_fourier_window_sequence(traj_std, L, K, gamma=1.0):
    T, B, N = traj_std.shape
    F_seq = fourier_encode_snapshot(traj_std, K=K)                  # (T, B, dF)
    dF = F_seq.shape[-1]

    t0 = L - 1
    t1 = T - 1
    t_vals = list(range(t0, t1 + 1))
    n_t = len(t_vals)

    Fw_seq = torch.empty(B, n_t, L * dF, device=traj_std.device, dtype=traj_std.dtype)
    w = gamma ** torch.arange(L, device=traj_std.device, dtype=traj_std.dtype)

    for j in range(B):
        for i, t in enumerate(t_vals):
            block = F_seq[t-L+1:t+1, j, :]                          # oldest->newest
            block = torch.flip(block, dims=[0])                     # newest->oldest
            block = block * w[:, None]                              # apply lag weights
            Fw_seq[j, i, :] = block.reshape(-1)

    return Fw_seq, t_vals

def rollout_state_space_fourier_window(
    S0_std, Omega, b, sigma, P, W, steps
):
    """
    S0_std: (B, d_window) initial latent window state in standardized latent coords
    returns:
        S_future_std: (steps+1, B, d_window)
    """
    device = S0_std.device
    dtype = S0_std.dtype
    B, d = S0_std.shape

    S_future_std = torch.empty(steps + 1, B, d, device=device, dtype=dtype)
    S_future_std[0] = S0_std

    s = S0_std
    for h in range(steps):
        ds = phi_batch(s, Omega, b, sigma, P) @ W
        s = s + ds
        S_future_std[h + 1] = s

    return S_future_std


def decode_fourier_window_rollout_std(S_future_std, S_mean, S_std, N, L, K, mu_x, sd_x):
    """
    S_future_std: (steps+1, B, d_window)
    S_mean, S_std: (1, d_window)
    mu_x, sd_x: (1,1,N)

    returns:
        X_hat:      (steps+1, B, N) decoded newest state in original scale
        X_hat_std:  (steps+1, B, N) decoded newest state in standardized scale
        W_hat_std:  (steps+1, B, L, dF) decoded Fourier windows
        Xw_hat_std: (steps+1, B, L, N) decoded snapshot windows in standardized scale
    """
    dF = 1 + 2 * K

    # undo latent standardization
    S_future = S_mean + S_std * S_future_std                      # (steps+1, B, L*dF)

    # reshape to window of Fourier snapshots
    W_hat_std = S_future.reshape(S_future.shape[0], S_future.shape[1], L, dF)   # (steps+1, B, L, dF)

    # decode each Fourier snapshot back to standardized physical space
    Xw_hat_std = fourier_decode_snapshot(W_hat_std, N=N, K=K)    # (steps+1, B, L, N)

    # newest state is index 0 because windows are stored newest->oldest
    X_hat_std = Xw_hat_std[:, :, 0, :]                           # (steps+1, B, N)
    X_hat = mu_x + sd_x * X_hat_std

    return X_hat, X_hat_std, W_hat_std, Xw_hat_std


# ============================================================
# CLEAN FOURIER-WINDOW PIPELINE
# ============================================================

# traj: (T, B, N)
T, B, N = traj.shape
T_train = T // 2
steps = 10
horizon = 1          # must be 1 for recursive rollout
L = 40

# choose Fourier truncation
K = 2               # per-snapshot latent dim dF = 1 + 2K
dF = 1 + 2 * K
d_window = L * dF

traj_train = traj[:T_train]   # (T_train, B, N)

# 1) standardize raw states on training block only
mu_x_train = traj_train.mean(dim=(0, 1), keepdim=True)              # (1,1,N)
sd_x_train = traj_train.std(dim=(0, 1), keepdim=True) + 1e-6        # (1,1,N)
traj_train_std = (traj_train - mu_x_train) / sd_x_train             # (T_train, B, N)

# 2) build Fourier-window latent sequence on training block
S_seq, t_vals = build_fourier_window_sequence(traj_train_std, L=L, K=K)
# S_seq shape: (B, n_t, d_window), where window i ends at t_vals[i]

B_, n_t, d_ = S_seq.shape
assert B_ == B and d_ == d_window
assert t_vals[-1] == T_train - 1

# 3) build one-step latent pairs
S_train = S_seq[:, :-1, :].reshape(-1, d_window)
dS = (S_seq[:, 1:, :] - S_seq[:, :-1, :]).reshape(-1, d_window)

# 4) standardize latent window-states
S_mean = S_train.mean(dim=0, keepdim=True)
S_std = S_train.std(dim=0, keepdim=True) + 1e-6

S_seq_std = (S_seq - S_mean) / S_std
S_train_std = S_seq_std[:, :-1, :].reshape(-1, d_window)
dS_std = (S_seq_std[:, 1:, :] - S_seq_std[:, :-1, :]).reshape(-1, d_window)

# 5) fit one-step latent increment model
dtype = traj.dtype
ell_s = median_ell(S_train_std)
sigma_s = 1.0
P_s = 512

Omega_s = torch.randn(P_s, d_window, device=traj.device, dtype=dtype) / ell_s
b_s = 2 * torch.pi * torch.rand((P_s,), device=traj.device, dtype=dtype)

W_s = solve_W(S_train_std, dS_std, Omega_s, b_s, sigma=sigma_s, P=P_s, lam=1e-2)

# 6) initialize from the last training window
# this corresponds to the window ending at time T_train - 1
S0_std = S_seq_std[:, -1, :]                                        # (B, d_window)

# 7) roll out in latent space
S_future_std = rollout_state_space_fourier_window(
    S0_std, Omega_s, b_s, sigma_s, P_s, W_s, steps
)

# 8) decode back to newest physical state
X_hat, X_hat_std, W_hat_std, Xw_hat_std = decode_fourier_window_rollout_std(
    S_future_std, S_mean, S_std, N=N, L=L, K=K, mu_x=mu_x_train, sd_x=sd_x_train
)

# 9) evaluate against held-out truth
# X_hat_std[0] corresponds to time T_train-1
# X_hat_std[1] corresponds to predicted time T_train
traj_future = traj[T_train:T_train + steps, :, :]                    # truth at times T_train ... T_train+steps-1
traj_future_std = (traj_future - mu_x_train) / sd_x_train

errs_fourier_window = []
for h in range(1, steps + 1):
    rel_err = (
        torch.linalg.norm(X_hat_std[h] - traj_future_std[h - 1]) /
        torch.linalg.norm(traj_future_std[h - 1])
    ).item()
    errs_fourier_window.append(rel_err)

print(errs_fourier_window)

[0.9721359610557556, 0.9720381498336792, 0.9719560146331787, 0.9718957543373108, 0.9718655347824097, 0.9718747138977051, 0.9719317555427551, 0.9720474481582642, 0.972231388092041, 0.9724923968315125]
